# Extraction & enrichissement en ligne (échantillon)

Ce notebook démontre la chaîne complète **en accès réel** aux ressources externes :
flux RSS ANSSI → JSON des bulletins → API MITRE (CVSS/CWE) → API FIRST (EPSS).

In [16]:
import pandas as pd
import feedparser
import requests
import re
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime
import time

CVE_PATTERN = r"CVE-\d{4}-\d{4,7}"
N_BULLETINS = 30        # taille de l'échantillon traité
DELAI_REQUETE = 2       # secondes entre deux appels externes (rate limiting)

## 1. Extraction des flux RSS (avis + alertes)

In [17]:
urls = {
    "Alertes": "https://www.cert.ssi.gouv.fr/alerte/feed/",
    "Avis": "https://www.cert.ssi.gouv.fr/avis/feed/",
}

bulletins = []
for type_bulletin, url in urls.items():
    rss_feed = feedparser.parse(url)
    for entry in rss_feed.entries:
        bulletins.append({
            "id_bulletin": entry.link.split("/")[-2],
            "titre": entry.title,
            "type": type_bulletin,
            "date_publication": entry.published,
            "lien": entry.link,
        })

print(f"{len(bulletins)} bulletins extraits.")
print("Exemple :", bulletins[0])

80 bulletins extraits.
Exemple : {'id_bulletin': 'CERTFR-2023-ALE-007', 'titre': '[MàJ] Vulnérabilité dans Zimbra Collaboration Suite (17 juillet 2023)', 'type': 'Alertes', 'date_publication': 'Mon, 17 Jul 2023 00:00:00 +0000', 'lien': 'https://www.cert.ssi.gouv.fr/alerte/CERTFR-2023-ALE-007/'}


## 2. Identification des CVE par bulletin

On ouvre le JSON de chaque bulletin (URL = lien + `json/`) et on récupère les CVE
par expression régulière, puis on déplie : **une ligne par CVE**.

In [18]:
def extraire_cve_bulletin(lien_bulletin):
    """Renvoie la liste des CVE d'un bulletin à partir de son JSON en ligne."""
    response = requests.get(lien_bulletin + "json/", timeout=10)
    response.raise_for_status()
    data = response.json()
    return list(set(re.findall(CVE_PATTERN, str(data))))


bulletins_cve = []
for b in bulletins[:N_BULLETINS]:
    try:
        for cve in extraire_cve_bulletin(b["lien"]):
            ligne = b.copy()
            ligne["cve_id"] = cve
            bulletins_cve.append(ligne)
    except Exception as e:
        print(f"Bulletin ignoré ({b['id_bulletin']}): {e}")
    time.sleep(DELAI_REQUETE)

print(f"{len(bulletins_cve)} couples (bulletin, CVE) extraits.")

58 couples (bulletin, CVE) extraits.


## 3. Enrichissement MITRE (CVSS, CWE, produit) et FIRST (EPSS)

In [20]:
def get_severity(score):
    """Convertit un score CVSS (0-10) en niveau de gravité CVSS v3."""
    try:
        score = float(score)
    except (TypeError, ValueError):
        return "Non disponible"
    if score == 0.0: return "None"
    elif score < 4.0: return "Low"
    elif score < 7.0: return "Medium"
    elif score < 9.0: return "High"
    return "Critical"


def get_mitre_details(cve_id):
    """Interroge l'API MITRE et renvoie un dict robuste aux champs manquants."""
    details = {k: "Non disponible" for k in
               ("cvss", "severity", "cwe", "description", "vendor", "product", "versions")}
    try:
        r = requests.get(f"https://cveawg.mitre.org/api/cve/{cve_id}", timeout=10)
        if r.status_code != 200:
            return details
        cna = r.json().get("containers", {}).get("cna", {})

        descriptions = cna.get("descriptions", [])
        if descriptions:
            details["description"] = descriptions[0].get("value", "Non disponible")

        for m in cna.get("metrics", []):
            if "cvssV3_1" in m:
                details["cvss"] = m["cvssV3_1"].get("baseScore", "Non disponible"); break
            elif "cvssV3_0" in m:
                details["cvss"] = m["cvssV3_0"].get("baseScore", "Non disponible"); break
        details["severity"] = get_severity(details["cvss"])

        problem_types = cna.get("problemTypes", [])
        if problem_types:
            details["cwe"] = problem_types[0].get("descriptions", [{}])[0].get("cweId", "Non disponible")

        affected = cna.get("affected", [])
        if affected:
            details["vendor"] = affected[0].get("vendor", "Non disponible")
            details["product"] = affected[0].get("product", "Non disponible")
            versions = [v.get("version", "") for v in affected[0].get("versions", [])
                        if v.get("status") == "affected"]
            if versions:
                details["versions"] = ", ".join(versions)
    except Exception as e:
        print(f"Erreur MITRE {cve_id}: {e}")
    return details


def get_epss_score(cve_id):
    """Interroge l'API FIRST et renvoie le score EPSS (probabilité d'exploitation)."""
    try:
        r = requests.get(f"https://api.first.org/data/v1/epss?cve={cve_id}", timeout=10)
        if r.status_code == 200:
            data = r.json().get("data", [])
            if data:
                return data[0].get("epss", "Non disponible")
    except Exception as e:
        print(f"Erreur EPSS {cve_id}: {e}")
    return "Non disponible"

In [24]:
donnees = []
total = min(len(bulletins_cve), N_BULLETINS)
for i, b in enumerate(bulletins_cve[:N_BULLETINS], start=1):
    cve_id = b["cve_id"]
    print(f"[{i}/{total}] {cve_id}")
    mitre = get_mitre_details(cve_id)
    epss = get_epss_score(cve_id)
    donnees.append({
        "ID ANSSI": b["id_bulletin"],
        "Titre ANSSI": b["titre"],
        "Type": b["type"],
        "Date": b["date_publication"],
        "CVE": cve_id,
        "CVSS": mitre["cvss"],
        "Base Severity": mitre["severity"],
        "CWE": mitre["cwe"],
        "EPSS": epss,
        "Lien": b["lien"],
        "Description": mitre["description"],
        "Éditeur": mitre["vendor"],
        "Produit": mitre["product"],
        "Versions affectées": mitre["versions"],
    })
    time.sleep(DELAI_REQUETE)

print("Enrichissement terminé.")

df_cve = pd.DataFrame(donnees)
df_cve["CVSS"] = pd.to_numeric(df_cve["CVSS"], errors="coerce")
df_cve["EPSS"] = pd.to_numeric(df_cve["EPSS"], errors="coerce")
df_cve["Date"] = pd.to_datetime(df_cve["Date"], format="mixed", errors="coerce")

display(df_cve.head())
df_cve.to_csv("sample_data_from_scrap.csv", index=False)
print("Échantillon sauvegardé dans 'sample_data_from_scrap.csv'.")

[1/30] CVE-2023-37580
[2/30] CVE-2023-3519
[3/30] CVE-2023-42114
[4/30] CVE-2023-42115
[5/30] CVE-2023-42118
[6/30] CVE-2023-42116
[7/30] CVE-2023-42117
[8/30] CVE-2023-42119
[9/30] CVE-2023-20273
[10/30] CVE-2023-20198
[11/30] CVE-2023-4966
[12/30] CVE-2023-50164
[13/30] CVE-2023-46805
[14/30] CVE-2024-22024
[15/30] CVE-2024-21888
[16/30] CVE-2024-21887
[17/30] CVE-2024-21893
[18/30] CVE-2024-0402
[19/30] CVE-2023-7028
[20/30] CVE-2024-21762
[21/30] CVE-2024-21413
[22/30] CVE-2024-3400
[23/30] CVE-2024-20353
[24/30] CVE-2024-20359
[25/30] CVE-2024-24919
[26/30] CVE-2024-6387
[27/30] CVE-2024-42010
[28/30] CVE-2024-42009
[29/30] CVE-2024-42008
[30/30] CVE-2024-40766
Enrichissement terminé.


,ID ANSSI,Titre ANSSI,Type,Date,CVE,CVSS,Base Severity,CWE,EPSS,Lien,Description,Éditeur,Produit,Versions affectées
0,CERTFR-2023-ALE-007,[MàJ] Vulnérabilité dans Zimbra Collaboration ...,Alertes,2023-07-17 00:00:00+00:00,CVE-2023-37580,NaN,Non disponible,Non disponible,0.59041,https://www.cert.ssi.gouv.fr/alerte/CERTFR-202...,Zimbra Collaboration (ZCS) 8 before 8.8.15 Pat...,n/a,n/a,n/a
1,CERTFR-2023-ALE-008,[MàJ] Vulnérabilité dans Citrix NetScaler ADC ...,Alertes,2023-07-19 00:00:00+00:00,CVE-2023-3519,9.8,Critical,CWE-94,0.99343,https://www.cert.ssi.gouv.fr/alerte/CERTFR-202...,Unauthenticated remote code execution,Citrix,NetScaler ADC,"13.1, 13.0, 13.1-FIPS, 12.1-FIPS, 12.1-NDcPP"
2,CERTFR-2023-ALE-010,Multiples vulnérabilités dans Exim (02 octobre...,Alertes,2023-10-02 00:00:00+00:00,CVE-2023-42114,3.7,Low,CWE-125,0.28084,https://www.cert.ssi.gouv.fr/alerte/CERTFR-202...,Exim NTLM Challenge Out-Of-Bounds Read Informa...,Exim,Exim,exim 4.95
3,CERTFR-2023-ALE-010,Multiples vulnérabilités dans Exim (02 octobre...,Alertes,2023-10-02 00:00:00+00:00,CVE-2023-42115,9.8,Critical,CWE-787,0.10042,https://www.cert.ssi.gouv.fr/alerte/CERTFR-202...,Exim AUTH Out-Of-Bounds Write Remote Code Exec...,Exim,Exim,exim 4.95
4,CERTFR-2023-ALE-010,Multiples vulnérabilités dans Exim (02 octobre...,Alertes,2023-10-02 00:00:00+00:00,CVE-2023-42118,7.5,High,CWE-191,0.51474,https://www.cert.ssi.gouv.fr/alerte/CERTFR-202...,Exim libspf2 Integer Underflow Remote Code Exe...,Exim,libspf2,exim 4.96-RC0-14-24b8ed847-XX


Échantillon sauvegardé dans 'sample_data_from_scrap.csv'.
